## Imports

In [ ]:
from pathlib import Path

import tiktoken
import torch

from nano_llm.loaders import create_dataloader_v1

## Variables

In [ ]:
TEXT_PATH = Path("..") / "the-verdict.txt"
VOCAB_SIZE = tiktoken.get_encoding("gpt2").n_vocab
CONTEXT_LENGTH = 4
EMBEDDING_DIM = 256

raw_text = TEXT_PATH.read_text(encoding="utf-8")

## Loader

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
second_batch = next(data_iter)

print(first_batch)
print("====")
print(second_batch)

## Token embeddings

In [ ]:
torch.manual_seed(123)

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
)
inputs, targets = next(iter(dataloader))

token_embedding_layer = torch.nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM)
token_embeddings = token_embedding_layer(inputs)

print("Token IDs:\n", inputs)
print("Token embeddings shape:", token_embeddings.shape)

## Positional embeddings

In [ ]:
pos_embedding_layer = torch.nn.Embedding(CONTEXT_LENGTH, EMBEDDING_DIM)
pos_embeddings = pos_embedding_layer(torch.arange(CONTEXT_LENGTH))

input_embeddings = token_embeddings + pos_embeddings

print("Positional embeddings shape:", pos_embeddings.shape)
print("Input embeddings shape:", input_embeddings.shape)

## Queries, keys and values

In [ ]:
query_layer = torch.nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM, bias=False)
key_layer = torch.nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM, bias=False)
value_layer = torch.nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM, bias=False)

queries = query_layer(input_embeddings)
keys = key_layer(input_embeddings)
values = value_layer(input_embeddings)

print("Queries shape:", queries.shape)
print("Keys shape:", keys.shape)
print("Values shape:", values.shape)

## Attention weights

In [ ]:
attention_scores = queries @ keys.mT
attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim=-1)

print("Attention weights shape:", attention_weights.shape)
print("Attention weights of the first sequence:\n", attention_weights[0])
print("Row sums:", attention_weights[0].sum(dim=-1))

## Context vectors

In [ ]:
context_vectors = attention_weights @ values

print("Context vectors shape:", context_vectors.shape)